# Machine Learning End-to-End Pipeline (Iris Dataset)
Complete classroom workflow: Data Inspection, Stratified Train/Test Split, Pipelines, Model Comparison, Confusion Matrix, 5-Fold Cross-Validation, K-Means Clustering, and Joblib/Pickle Persistence.

### 1. Imports and Dataset

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.base import clone
import joblib, pickle

data = load_iris(as_frame=True)
X = data.data.copy()
y = data.target.copy() # integer-encoded species: 0, 1, 2
class_names = list(data.target_names)
print("Shape:", X.shape)
print("Features:", list(X.columns))
print("Species:", class_names)
print("Samples per species:\n", y.map(dict(enumerate(class_names))).value_counts())

### 2. Inspect Data Quality

In [ ]:
display(X.head())
print("Types:\n", X.dtypes.value_counts())
print("Missing values:", int(X.isna().sum().sum()))
print("Duplicate feature rows:", int(X.duplicated().sum()))
display(y.map(dict(enumerate(class_names))).value_counts().rename("sample_count"))

### 3. Train/Test Split (Do Splitting)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training shape:", X_train.shape, "Testing shape:", X_test.shape)
print("Training class counts:\n", y_train.map(dict(enumerate(class_names))).value_counts())
print("Testing class counts:\n", y_test.map(dict(enumerate(class_names))).value_counts())

### 4. Build Pipelines and Candidate Models

In [ ]:
def make_pipeline(model, scale=True):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    # Select 3 of the 4 measurements. Selection is fitted only on training data.
    steps.append(("select", SelectKBest(score_func=mutual_info_classif, k=3)))
    steps.append(("model", model))
    return Pipeline(steps)

models = {
    "Logistic Regression": make_pipeline(LogisticRegression(max_iter=3000)),
    "Decision Tree": make_pipeline(DecisionTreeClassifier(max_depth=4, min_samples_leaf=3, random_state=42), scale=False),
    "Random Forest": make_pipeline(RandomForestClassifier(n_estimators=200, min_samples_leaf=2, random_state=42), scale=False),
    "SVM (RBF)": make_pipeline(SVC(kernel="rbf", C=1.0, probability=True, random_state=42))
}
print(list(models))

### 5. Train and Compare

In [ ]:
results, fitted = [], {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name] = pipe
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision_macro": precision_score(y_test, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, pred, average="macro", zero_division=0)
    })
results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False)
display(results_df.round(3))

### 6. Confusion Matrix and Classification Report

In [ ]:
best_name = results_df.iloc[0]["model"]
best_model = fitted[best_name]
pred = best_model.predict(X_test)
print("Model:", best_name)
print(classification_report(y_test, pred, target_names=class_names, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=class_names, xticks_rotation=25)
plt.title(best_name)
plt.tight_layout()
plt.show()

### 7. Cross Validation within Training Data

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clone(models["Logistic Regression"]), X_train, y_train, cv=cv, scoring="f1_macro")
print("Fold macro-F1:", np.round(scores, 3))
print("Mean macro-F1:", round(scores.mean(), 3))

### 8. K-Means Clustering

In [ ]:
prep = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
X_train_scaled = prep.fit_transform(X_train)
km = KMeans(n_clusters=3, n_init=20, random_state=42)
clusters = km.fit_predict(X_train_scaled)
print("Silhouette:", round(silhouette_score(X_train_scaled, clusters), 3))
display(pd.crosstab(
    pd.Series(clusters, name="cluster"),
    pd.Series(y_train.map(dict(enumerate(class_names))).reset_index(drop=True), name="known species (for interpretation only)")
))

### 9. Save and Reload the Complete Pipeline (Joblib & Pickle)

In [ ]:
joblib.dump(best_model, "iris_demo_pipeline.joblib")
loaded = joblib.load("iris_demo_pipeline.joblib")
sample = X_test.iloc[[0]]
print("Before:", class_names[int(best_model.predict(sample)[0])])
print("After reload:", class_names[int(loaded.predict(sample)[0])])

# Pickle alternative (only load trusted artifacts)
with open("iris_demo_pipeline.pkl", "wb") as f:
    pickle.dump(best_model, f)
with open("iris_demo_pipeline.pkl", "rb") as f:
    loaded_pickle = pickle.load(f)
print("Pickle reload:", class_names[int(loaded_pickle.predict(sample)[0])])